# Automated Quality Control with Transfer Learning – Manufacturing Defect Detection

This notebook demonstrates transfer learning using ResNet50 for binary defect detection in manufacturing.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report
import os

print(tf.__version__)

## Discovery Phase: Data Preparation

In [ ]:
# Assume dataset is organized as:
# data/
#   train/
#     good/
#     defect/
#   test/
#     good/
#     defect/

# For demonstration, you can download a dataset like NEU or MVTec and organize accordingly.
# Here we use ImageDataGenerator for augmentation.

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

test_datagen = ImageDataGenerator(rescale=1./255)

# Replace with your dataset path
train_generator = train_datagen.flow_from_directory(
    'data/train',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    'data/train',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

test_generator = test_datagen.flow_from_directory(
    'data/test',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

## Technical Phase: Model Construction (Brain Swap)

In [ ]:
# Load pre-trained ResNet50
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the base model
base_model.trainable = False

# Add custom head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# Train the model
history = model.fit(
    train_generator,
    epochs=10,
    validation_data=validation_generator
)

In [ ]:
# Plot training curves
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.legend()
plt.title('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Loss')
plt.show()

## Action Phase: Inference and Decision Logic

In [ ]:
# Inference on new image
def predict_defect(image_path):
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=(224, 224))
    img_array = tf.keras.preprocessing.image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_array)[0][0]
    return prediction

# Example
# prob = predict_defect('path/to/new_image.jpg')
# print(f'Defect Probability: {prob:.2%}')

In [ ]:
# Automated decision logic
def factory_decision(prob, threshold=0.85):
    if prob >= threshold:
        return "Defect detected: Trigger robotic arm to reject/remove product."
    else:
        return "Good product: Proceed with assembly/packaging."

# Test on batch
test_generator.reset()
predictions = model.predict(test_generator)
predicted_classes = (predictions > 0.5).astype(int).flatten()
true_classes = test_generator.classes

print(classification_report(true_classes, predicted_classes, target_names=['Good', 'Defect']))

## Explanation of Key Choices

- **GlobalAveragePooling2D vs Flatten**: GAP reduces parameters dramatically (no dense connections to all features), preserves spatial hierarchy better, and helps mitigate overfitting. Ideal for transfer learning.

Limitations: Requires sufficient data; fine-tuning base layers may improve performance further. Threshold tuning based on business cost of false positives/negatives.